# Exploración de Capacitaciones — PISST
**Objetivo:** Explorar asistencias y evaluaciones para validar la lógica de `analizar_capacitaciones()`.

Este notebook es de solo lectura — nunca escribe en la BD.

## 1. Configuración e imports

In [ ]:
import os
import sys

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

sys.path.insert(0, os.path.abspath('..'))
load_dotenv('../.env')

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

print('Pandas:', pd.__version__, '| NumPy:', np.__version__)

## 2. Conexión a la BD

In [ ]:
DATABASE_URL = os.getenv('DATABASE_URL')
engine = create_engine(DATABASE_URL)

with engine.connect() as conn:
    total_cap = conn.execute(text('SELECT COUNT(*) FROM capacitaciones')).scalar()
    total_ses = conn.execute(text('SELECT COUNT(*) FROM sesiones_capacitacion')).scalar()
    total_asi = conn.execute(text('SELECT COUNT(*) FROM asistencias')).scalar()
    total_resp = conn.execute(text('SELECT COUNT(*) FROM respuestas_empleado')).scalar()
    print(f'Capacitaciones: {total_cap} | Sesiones: {total_ses} | Asistencias: {total_asi} | Respuestas: {total_resp}')

## 3. Carga de datos (Ingesta)

In [ ]:
# Asistencias con datos de sesión y capacitación
query_asistencias = """
SELECT
    a.empleado_id,
    u.nombre         AS empleado_nombre,
    a.estado         AS asistencia_estado,
    sc.estado        AS sesion_estado,
    c.empresa_id
FROM asistencias a
JOIN sesiones_capacitacion sc ON sc.id = a.sesion_id
JOIN capacitaciones c         ON c.id  = sc.capacitacion_id
JOIN users u                  ON u.id  = a.empleado_id
"""

# Resultados de evaluaciones
query_evaluaciones = """
SELECT
    re.empleado_id,
    re.aprobado,
    re.puntaje_final,
    c.empresa_id
FROM respuestas_empleado re
JOIN evaluaciones ev          ON ev.id  = re.evaluacion_id
JOIN sesiones_capacitacion sc ON sc.id  = ev.sesion_id
JOIN capacitaciones c         ON c.id   = sc.capacitacion_id
"""

# Capacitaciones sin sesión realizada
query_sin_sesion = """
SELECT c.id, c.titulo, c.empresa_id
FROM capacitaciones c
WHERE c.activo = TRUE
  AND NOT EXISTS (
    SELECT 1 FROM sesiones_capacitacion sc
    WHERE sc.capacitacion_id = c.id AND sc.estado = 'realizada'
  )
"""

df_asistencias = pd.read_sql(query_asistencias, engine)
df_evaluaciones = pd.read_sql(query_evaluaciones, engine)
df_sin_sesion = pd.read_sql(query_sin_sesion, engine)

print(f'Asistencias: {len(df_asistencias)} | Evaluaciones: {len(df_evaluaciones)} | Cap sin sesión realizada: {len(df_sin_sesion)}')
df_asistencias.head()

## 4. Inspección estructural

In [ ]:
print('Shape asistencias:', df_asistencias.shape)
df_asistencias.info()
print()
print('Estados de asistencia:', df_asistencias['asistencia_estado'].unique())
print('Estados de sesión:', df_asistencias['sesion_estado'].unique())

## 5. Análisis descriptivo

In [ ]:
print('Distribución de asistencia:')
print(df_asistencias['asistencia_estado'].value_counts())
print()

if len(df_evaluaciones) > 0:
    print('Distribución aprobado/reprobado:')
    print(df_evaluaciones['aprobado'].value_counts())
    print()
    print('Estadísticas de puntaje:')
    print(df_evaluaciones['puntaje_final'].describe())

## 6. Transformaciones y cálculos (lógica del servicio)

In [ ]:
# --- Tasa de aprobación ---
if len(df_evaluaciones) == 0:
    total_evaluaciones = 0
    tasa_aprobacion = 0.0
else:
    total_evaluaciones = len(df_evaluaciones)
    tasa_aprobacion = round(df_evaluaciones['aprobado'].mean() * 100, 1)

print(f'Total evaluaciones: {total_evaluaciones}')
print(f'Tasa de aprobación: {tasa_aprobacion}%')

In [ ]:
# --- Asistencia promedio y alertas con NumPy ---
if len(df_asistencias) == 0:
    asistencia_promedio = 0.0
    alertas = []
else:
    # Solo contar sesiones de capacitaciones que ya se realizaron
    df_realizadas = df_asistencias[df_asistencias['sesion_estado'] == 'realizada']

    if len(df_realizadas) == 0:
        asistencia_promedio = 0.0
        alertas = []
    else:
        # % asistencia por empleado (presente = 1, ausente/justificado = 0)
        df_realizadas = df_realizadas.copy()
        df_realizadas['presente'] = np.where(df_realizadas['asistencia_estado'] == 'presente', 1, 0)

        por_empleado = df_realizadas.groupby(['empleado_id', 'empleado_nombre'])['presente'].agg(['sum', 'count'])
        por_empleado['pct_asistencia'] = round(por_empleado['sum'] / por_empleado['count'] * 100, 1)
        por_empleado = por_empleado.reset_index()

        asistencia_promedio = round(por_empleado['pct_asistencia'].mean(), 1)

        # Alertas: empleados con asistencia < 80%
        df_alertas = por_empleado[por_empleado['pct_asistencia'] < 80]
        alertas = df_alertas[['empleado_id', 'empleado_nombre', 'pct_asistencia']].rename(
            columns={'empleado_nombre': 'nombre', 'pct_asistencia': 'asistencia_pct'}
        ).to_dict('records')

        print('Por empleado (asistencia):')
        print(por_empleado[['empleado_nombre', 'pct_asistencia']].to_string(index=False))

print(f'\nAsistencia promedio: {asistencia_promedio}%')
print(f'Alertas (< 80%): {len(alertas)}')

In [ ]:
# Resultado final
resultado = {
    'total_evaluaciones': total_evaluaciones,
    'tasa_aprobacion_pct': tasa_aprobacion,
    'asistencia_promedio_pct': asistencia_promedio,
    'alertas_asistencia': alertas,
    'capacitaciones_sin_sesion_realizada': len(df_sin_sesion),
}

import json
print(json.dumps(resultado, indent=2, ensure_ascii=False, default=str))

## 7. Exportación de hallazgos

In [ ]:
os.makedirs('../data/processed', exist_ok=True)

if len(df_asistencias) > 0:
    df_asistencias.to_csv('../data/processed/asistencias_exploradas.csv', index=False)
if len(df_evaluaciones) > 0:
    df_evaluaciones.to_csv('../data/processed/evaluaciones_exploradas.csv', index=False)
print('Exportado a data/processed/')

## 8. Conclusiones para el servicio

- `analizar_capacitaciones()` necesita 3 queries: asistencias, evaluaciones, capacitaciones sin sesión realizada
- Usar `np.where` para convertir estado → 1/0 antes de calcular el promedio por empleado
- Las alertas de asistencia solo se generan sobre sesiones con `estado == 'realizada'`
- La tasa de aprobación es `aprobado.mean() * 100` sobre todas las respuestas del empleado
- El umbral de alerta es `< 80%` de asistencia